## Attention Exploration (14 points)

Multi-head self-attention is the core modeling component of Transformers. In this question, we’ll get some practice working with the self-attention equations, and motivate why multi-headed self-attention can be preferable to single-headed self-attention.

Recall that attention can be viewed as an operation on a *query* vector $q \in \mathbb{R}^d$, a set of *value* vectors
$\{v_1,...,v_n\},v_i \in \mathbb{R}^d$, and a set of *key* vectors $\{k_1,...,k_n\},k_i \in \mathbb{R}^d$, specified as follows:

$$
c = \sum_{i=1}^{n} v_i \alpha_i \quad (1)
$$

$$
\alpha_i =
\frac{\exp(k_i^\top q)}
{\sum_{j=1}^{n} \exp(k_j^\top q)} \quad (2)
$$

with $\alpha= {\alpha_1,...,\alpha_n}$ termed the "attention weights". Observe that the output $c\in\mathbb{R}^d$ is an average over the value vectors weighted with respect to $\alpha$.

### (a) (3 points) Copying in attention.

One advantage of attention is that it’s particularly easy to "copy" a value vector to the output c. In this problem, we’ll motivate why this is the case.

i. (2 points) The distribution $\alpha$ is typically relatively "diffuse"; the probability mass is spread out between many different $\alpha_i$. However, this is not always the case. Describe (in one sentence) under what conditions the categorical distribution $\alpha$ puts almost all of its weight on some $\alpha_j$, where $j \in\{1,...,n\}$ (i.e. $\alpha_j \gg \sum_{i\not=j}\alpha_i$). What must be true about the query $q$ and/or the keys $\{k_1,...,k_n\}$?


Here's the main idea. Suppose $k_i$ is just the set of unit orthonormal vectors. Suppose we have a one-hot-vector with 1 in the $j$-th position and 0 elsewhere. This does not quite work due to the fact that $exp(0) = 1$, but we may put some large constant $C \gg 1$ in the $j$-th position instead. The general answer follows.

The attention distribution puts almost all its weight on $\alpha_j$ when the query $q$ has a much larger dot product with key $k_j$ than with any other key; that is, $k_j^\top q \gg k_i^\top q$ for all $i\ne j$. In this case, $\exp(k_j^\top q)$ dominates the denominator of the softmax, so $\alpha_j\approx 1$ and $\alpha_i\approx 0$ for $i\ne j$. This can happen when $q$ and $k_j$ are strongly aligned while the other keys are not.

---


ii. (1 point) Under the conditions you gave in (i), describe the output c.

In this case $c \approx v_j$ since $\alpha_j \approx 1$ and $\alpha_i \approx 0$ for all $i \ne j$.

### (b) (2 points) An average of two.

Instead of focusing on just one vector $v_j$, a Transformer model might want to incorporate information from multiple source vectors. Consider the case where we instead want to incorporate information from two vectors $v_a$ and $v_b$, with corresponding key vectors $k_a$ and $k_b$. Assume that (1) all key vectors are orthogonal, so $k_i^\top k_j = 0$ for all $i \ne j$; and (2) all key vectors have norm 1. Find an expression for a query vector $q$ such that $c \approx \frac{1}{2} (v_a + v_b)$, and justify your answer. (Recall what you learned in part (a).)  
Hint: while the softmax function will never exactly average the two vectors, you can get close by using a large scalar
multiple in the expression.


WLOG, assume that $k_i$ are just unit vectors with $k_{ii} = 1$ and $k_{ij} = 0$ for all $i \ne j$. Then we may take $q_a = q_b = C$, where $C$ is some large number and the other components of $q$ are 0. This ensures that the attention weights for $v_a$ and $v_b$ are approximately equal and dominate the softmax, giving $c \approx \frac{1}{2} (v_a + v_b)$.

#### Answer from LUNA

Your answer is essentially correct. The cleanest expression is

$$
q=C(k_a+k_b),
$$

where $C>0$ is large. Then, using orthonormal keys,

$$
k_a^\top q=C,\qquad k_b^\top q=C,
$$

while every other key has score $0$. Therefore,

$$
\alpha_a=\alpha_b
=\frac{e^C}{2e^C+(n-2)}
\longrightarrow \frac12
$$

as $C\to\infty$, and the remaining attention weights approach $0$. Hence,

$$
c\approx \frac12v_a+\frac12v_b.
$$

Your coordinate version is valid if the keys are identified with the standard basis, but change $k_{ii}$ and $k_{ij}$ to statements about the **components** of $k_i$, and use $a,b$ consistently instead of $i,j$.

### (c) (5 points) Drawbacks of single-headed attention

In the previous part, we saw how it was possible for a single-headed attention to focus equally on two values. The same concept could easily be extended to any subset of values. In this question we’ll see why it’s not a practical solution.

Consider a set of key vectors $\{k_1,...,k_n\}$ that are now randomly sampled, $k_i \sim N(\mu_i,\Sigma_i)$, where the means $\mu_i \in \mathbb{R}^d$ are known to you, but the covariances $\Sigma_i$ are unknown (unless specified otherwise in the question). Further, assume that the means $\mu_i$ are all perpendicular; $\mu_i^\top \mu_j = 0$ if $i\neq j$, and unit norm, $\|\mu_i\|= 1$.

---

i. (2 points) Assume that the covariance matrices are $\Sigma_i = \alpha I, \forall i \in \{1,2,...,n\}$, for vanishingly small $\alpha$. Design a query $q$ in terms of the $\mu_i$ such that as before, $c\approx\frac{1}{2} (v_a+ v_b)$, and provide a brief argument as to why it works.

My initial idea was to sample the query from $q_a \sim N(\mu_a,\Sigma_i)$ and $q_b \sim N(\mu_b,\Sigma_i)$ and then take a sum $q = C (q_a + q_b)$. And this would work since with a small $\alpha$ this $q$ will be approximately orthogonal to $k_i$ for all $i \notin \{a,b\}$.

But in the exercise they ask about $q$ in terms of the means $\mu_a$ and $\mu_b$, actually. SO instead of sampleing we may jjust take a sum of these means: $q = C (\mu_a + \mu_b)$, where $C>0$ is large. This works for the same reason as before: $q$ will have a large inner product with $k_a$ and $k_b$, and approximately zero with all other $k_i$.

---

ii. (3 points) Though single-headed attention is resistant to small perturbations in the keys, some types of larger perturbations may pose a bigger issue. In some cases, one key vector ka may be larger or smaller in norm than the others, while still pointing in the same direction as $\mu_a$.

As an example, let us consider a covariance for item a as $\Sigma_a = \alpha I+ \frac{1}{2} (\mu_a\mu_a^\top)$ for vanishingly small $\alpha$ (as shown in figure 1). This causes $k_a$ to point in roughly the same direction as $\mu_a$, but with large variances in magnitude. Further, let $\Sigma_i = \alpha I$ for all $i\not= a$.

When you sample $\{k_1,...,k_n\}$ multiple times, and use the $q$ vector that you defined in part i.,
what do you expect the vector $c$ will look like qualitatively for different samples? Think about
how it differs from part (i) and how $c$’s variance would be affected.

So again, $q = C (\mu_a + \mu_b)$. We still have $k_i^T q = 0$ for all $i \notin \{a,b\}$. We also still have $k_b^T q = C k_b^T (\mu_a + \mu_b) = C k_b^T \mu_b = C \|\mu_b\|^2 = C$ since $k_b$ is approximately aligned with $\mu_b$ due to small variance and has the same norm as $\mu_b$, which is 1.

But what about $k_a^T q$? Since $k_a$ may have a larger or smaller norm than $\mu_a$ while still pointing in roughly the same direction, $k_a^T q = C k_a^T (\mu_a + \mu_b) \approx C k_a^T \mu_a = C \|k_a\| \|\mu_a\| \cos\theta \approx C \|k_a\|$, where $\theta$ is the small angle between $k_a$ and $\mu_a$. Therefore, $k_a^T q$ could vary significantly depending on the sampled magnitude of $k_a$.

#### Answer from LUNA

Yes, your main reasoning is correct: the perturbation in $k_a$ changes its **norm in the $\mu_a$ direction**, so the score $k_a^\top q$ can vary substantially even though $q$ remains fixed.

A few corrections:

1. For sampled keys, $k_i^\top q$ is only approximately zero for $i\notin\{a,b\}$.
2. $k_b^\top q$ is approximately $C$, not exactly $C$, because $k_b$ also has small noise.
3. The important issue with $k_a$ is best expressed through its projection onto $\mu_a$:

$$
k_a \approx (1+z)\mu_a,
\qquad z\sim N\left(0,\frac12\right),
$$

ignoring the vanishingly small orthogonal noise. Therefore,

$$
k_a^\top q
\approx C(1+z),
$$

whereas

$$
k_b^\top q\approx C.
$$

Thus, depending on the sample:

- if $z>0$, $k_a^\top q$ may exceed $k_b^\top q$, so $c$ moves toward $v_a$;
- if $z<0$, $k_b^\top q$ may dominate, so $c$ moves toward $v_b$;
- if $z\approx0$, the output is close to $\frac12(v_a+v_b)$.

Because the softmax exponentially amplifies score differences, $c$ can vary greatly across samples. Consequently, compared with part (i), the output has substantially larger variance and may switch between values near $v_a$, values near $v_b$, and intermediate mixtures.

Your geometric explanation is good, but saying that $k_a$ “points in roughly the same direction” can be misleading when its longitudinal coefficient $1+z$ becomes negative or close to zero. The projection formulation above is more precise.

### (d) (3 points) Benefits of multi-headed attention

Now we’ll see some of the power of multi-headed attention. We’ll consider a simple version of multi-headed attention which is identical to single-headed self-attention as we’ve presented it, except two query vectors ($q_1$ and $q_2$) are defined, which leads to a pair of vectors ($c_1$ and $c_2$), each the output of single-headed attention given its respective query vector. The final output of the multi-headed attention is their average, $1/2 (c_1 + c_2)$.

As in question 1(c), consider a set of key vectors ${k_1,...,k_n}$ that are randomly sampled, $k_i ∼ N(µ_i,Σ_i)$, where the means $µ_i$ are known to you, but the covariances $Σ_i$ are unknown. Also as before, assume that the means $µ_i$ are mutually orthogonal; $µ_i^\top µ_j = 0$ if $i \ne j$, and unit norm, $\|µ_i\|= 1$.

---

i. (1 point) Assume that the covariance matrices are $Σ_i = αI$, for vanishingly small $\alpha$. Design $q_1$ and $q_2$ in terms of $µ_i$ such that $c$ is approximately equal to $\frac{1}{2} (v_a + v_b)$. Note that $q_1$ and $q_2$ should have different expressions

As above let's take $q_1 = C µ_a, \quad q_2 = C µ_b$, where $C$ is a large constant $C \gg 1$. All $k_i$ are approximately equal to their means $µ_i$ due to the vanishingly small covariance, so $k_i^T q_1 \approx C µ_i^T µ_a = C \delta_{ia}$ and $k_i^T q_2 \approx C µ_i^T µ_b = C \delta_{ib}$, where $\delta_{ij}$ is the Kronecker delta. 

Therefore, the attention weights will be approximately one for the corresponding keys and zero for all others, leading to $c_1 \approx v_a$ and $c_2 \approx v_b$, and thus $c \approx \frac{1}{2} (v_a + v_b)$.

---

ii. (2 points) Assume that the covariance matrices are $Σ_a = \alpha I+ \frac{1}{2} (\mu_a \mu_a^⊤)$ for vanishingly small
$\alpha$, and $\Sigma_i = \alpha I$ for all $i \neq a$. Take the query vectors $q_1$ and $q_2$ that you designed in part
i. What, qualitatively, do you expect the output c to look like across different samples of the key vectors? Explain briefly in terms of variance in $c_1$ and $c_2$. You can ignore cases in which $k_a^⊤ q_i <0$.

First of all, we still have $c_2 \approx v_b$. 

What about $c_1$? Well, the direction of $k_a$ is still the same as $\mu_a$, so orthogonality will hold and we get 

$$k_a^T q_1 \approx C k_a^T µ_a = C \|k_a\| \|\mu_a\| cos \theta \approx C \|k_a\| = s_a$$

where $\theta$ is the (small) angle between $k_a$ and $\mu_a$. Key condition: If score $s_a$ is still large enough, we will get the same result as in part i, namely $c_1 \approx v_a$.

---

### (e) multi-headed attention advantages

(1 point) Based on part (d), briefly summarize how multi-headed attention overcomes the drawbacks of single-headed attention that you identified in part (c).

A single-headed attention mechanism can *in principle* focus on two or more aspects of the input, but they are not independent and may interfere with each other. One of them can be skewed or dominated by the other, leading to suboptimal attention distribution.

Multi-headed attention, on the other hand, allows the model to attend to different parts of the input *in parallel*, *independently*, thereby overcoming this limitation.

#### Answer from LUNA

Your answer is correct in general, but it would be stronger if it explicitly connected to part (d): a perturbation in one key can distort a single head’s combined attention, whereas it affects mainly one multi-head component.

A polished version:

> Single-headed attention must use one query and one softmax distribution to attend to multiple values. Consequently, a perturbation in one key can change the relative attention weights and interfere with attention to the other values. Multi-headed attention uses separate queries and softmax distributions, allowing different heads to focus on different values independently. Thus, a perturbation affecting one head has much less effect on the other head, making the final averaged output more robust.

This directly explains the benefit shown in the preceding parts.